# pfngouin Tutorial: Covariate-Adjusted Hypothesis Tests

This notebook shows how `pfngouin` mirrors the [pingouin](https://pingouin-stats.org/) API while adding **CUPED variance reduction** via covariate adjustment.

CUPED (Controlled-experiment Using Pre-Experiment Data) adjusts outcomes for covariates that correlate with the metric, reducing noise and increasing statistical power — so you can detect smaller effects or reach significance with fewer users.

We cover:
- `ttest` — Welch t-test
- `mwu` — Mann-Whitney U test

In [1]:
import sys

sys.path.insert(0, "..")

import pingouin as pg

import pfngouin as pfg

print(f"pfngouin version: {pfg.__version__}")

pfngouin version: 0.1.0


## 1. Simulate an A/B experiment

We use `make_experiment_data` from `data_generation.py` to produce a randomized A/B test where:
- Users have three pre-experiment covariates with nonlinear relationships to the outcome
- The treatment has a small true additive effect (`effect=1.0` by default)

This setup is designed to be challenging for linear models, making nonlinear covariate models (e.g. `TabPFNModel`) particularly valuable.

In [11]:
from pfngouin.datasets import make_experiment_data

data = make_experiment_data(N=2_000, effect=0.9, seed=42)
control   = data["control"]
treatment = data["treatment"]
X_ctrl    = data["X_ctrl"]
X_trt     = data["X_trt"]

print(f"Control   — n: {len(control)},   mean: {control.mean():.3f}, std: {control.std():.3f}")
print(f"Treatment — n: {len(treatment)}, mean: {treatment.mean():.3f}, std: {treatment.std():.3f}")
print(f"Observed difference: {treatment.mean() - control.mean():.3f}  (true effect: 0.9)")

Control   — n: 1007,   mean: 14.516, std: 10.814
Treatment — n: 993, mean: 15.180, std: 10.318
Observed difference: 0.664  (true effect: 0.9)


## 2. T-test

**Null hypothesis:** the means of the two groups are equal (μ_control = μ_treatment).

### 2a. Plain pingouin (no covariate adjustment)

In [12]:
result_plain_ttest = pg.ttest(treatment, control)
result_plain_ttest

,T,dof,alternative,p_val,CI95,cohen_d,power,BF10
T_test,1.404255,1995.831132,two-sided,0.160398,"[-0.26, 1.59]",0.062781,0.289202,0.134


### 2b. pfngouin with LinearModel

`LinearModel` applies OLS regression on the covariates. It is fast and works well when the covariate–outcome relationship is approximately linear.

In [13]:
result_linear_ttest = pfg.ttest(
    control,
    treatment,
    covariates_control=X_ctrl,
    covariates_treatment=X_trt,
    model=pfg.LinearModel(),
)
result_linear_ttest

,T,dof,alternative,p_val,CI95,cohen_d,power,BF10,var_reduction
T_test,1.468056,1997.969729,two-sided,0.142246,"[-0.19, 1.34]",0.065647,0.311383,0.146,0.3167


### 2c. pfngouin with TabPFNModel

`TabPFNModel` is a foundation model for tabular data. It captures nonlinear covariate–outcome relationships without feature engineering, which suits the nonlinear data generated here.

In [ ]:
result_tabpfn_ttest = pfg.ttest(
    control,
    treatment,
    covariates_control=X_ctrl,
    covariates_treatment=X_trt,
    model=pfg.TabPFNModel(), # default model
)
result_tabpfn_ttest

,T,dof,alternative,p_val,CI95,cohen_d,power,BF10,var_reduction
T_test,1.445436,1997.996302,two-sided,0.148492,"[-0.19, 1.28]",0.064638,0.303485,0.142,0.3717


### 2d. Comparison

In [15]:
import pandas as pd


def ci_width(result):
    ci = result["CI95"].values[0]
    return ci[1] - ci[0]

compare_ttest = pd.DataFrame({
    "method":        ["pingouin (no adjustment)", "pfngouin — LinearModel", "pfngouin — TabPFNModel"],
    "p-value":       [result_plain_ttest["p_val"].values[0],
                      result_linear_ttest["p_val"].values[0],
                      result_tabpfn_ttest["p_val"].values[0]],
    "CI width":      [ci_width(result_plain_ttest),
                      ci_width(result_linear_ttest),
                      ci_width(result_tabpfn_ttest)],
    "var_reduction": ["—",
                      f"{result_linear_ttest['var_reduction'].values[0]:.1%}",
                      f"{result_tabpfn_ttest['var_reduction'].values[0]:.1%}"],
})
compare_ttest.set_index("method")

,p-value,CI width,var_reduction
method,,,
pingouin (no adjustment),0.160398,1.85,—
pfngouin — LinearModel,0.142246,1.53,31.7%
pfngouin — TabPFNModel,0.148492,1.47,37.2%


## 3. Mann-Whitney U test

**Null hypothesis:** a randomly selected value from the treatment group is equally likely to be greater or less than a randomly selected value from the control group — i.e. P(treatment > control) = 0.5.

MWU is a rank-based, non-parametric alternative to the t-test. It makes no normality assumption and is more robust to outliers and skewed distributions.

### 3a. Plain pingouin

In [16]:
result_plain_mwu = pg.mwu(treatment, control)
result_plain_mwu

,U_val,alternative,p_val,RBC,CLES
MWU,517484.0,two-sided,0.175144,0.035019,0.517509


### 3b. pfngouin with LinearModel

In [17]:
result_linear_mwu = pfg.mwu(
    control,
    treatment,
    covariates_control=X_ctrl,
    covariates_treatment=X_trt,
    model=pfg.LinearModel(),
)
result_linear_mwu

,U_val,alternative,p_val,RBC,CLES,var_reduction
MWU,518378.0,two-sided,0.154131,0.036807,0.518403,0.3167


### 3c. pfngouin with TabPFNModel

In [ ]:
result_tabpfn_mwu = pfg.mwu(
    control,
    treatment,
    covariates_control=X_ctrl,
    covariates_treatment=X_trt,
    model=pfg.TabPFNModel(), # default model
)
result_tabpfn_mwu

,U_val,alternative,p_val,RBC,CLES,var_reduction
MWU,518789.0,two-sided,0.145138,0.037629,0.518814,0.3718


### 3d. Comparison

In [20]:
compare_mwu = pd.DataFrame({
    "method":        ["pingouin (no adjustment)", "pfngouin — LinearModel", "pfngouin — TabPFNModel"],
    "p-value":       [result_plain_mwu["p_val"].values[0],
                      result_linear_mwu["p_val"].values[0],
                      result_tabpfn_mwu["p_val"].values[0]],
    "CLES":          [result_plain_mwu["CLES"].values[0],
                      result_linear_mwu["CLES"].values[0],
                      result_tabpfn_mwu["CLES"].values[0]],
    "var_reduction": ["—",
                      f"{result_linear_mwu['var_reduction'].values[0]:.1%}",
                      f"{result_tabpfn_mwu['var_reduction'].values[0]:.1%}"],
})
compare_mwu.set_index("method")

,p-value,CLES,var_reduction
method,,,
pingouin (no adjustment),0.175144,0.517509,—
pfngouin — LinearModel,0.154131,0.518403,31.7%
pfngouin — TabPFNModel,0.145138,0.518814,37.2%
